# Lesson 6.2 · How many components to keep

Reducing the dimensions means choosing `k`. There is no absolutely right `k`: it depends on what you want to do (reconstruct, visualize, feed a model). This lesson shows the three criteria in use and their limits.

**What you will learn**

- Scree plot, cumulative variance and the practical rules (threshold on the variance, elbow).
- Why the best choice for a supervised model is cross-validation.
- What `PCA(n_components=0.95)` does.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)                        # 1797 8x8 images = 64 pixels
pca = PCA().fit(X)
cumulative = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(range(1, 65), pca.explained_variance_ratio_, "o-", ms=3); ax[0].set(title="scree plot", xlabel="component", ylabel="share of variance")
ax[1].plot(range(1, 65), cumulative, "o-", ms=3); ax[1].axhline(0.95, color="r", ls=":"); ax[1].set(title="cumulative variance", xlabel="number of components")
plt.tight_layout(); plt.show()
for threshold in [0.8, 0.9, 0.95, 0.99]:
    print(f"components to explain {threshold:.0%} of the variance: {int(np.searchsorted(cumulative, threshold)) + 1}")

## Three rules, three answers

- **Threshold on the cumulative variance** (90%, 95%, 99%): simple, but arbitrary. With `PCA(n_components=0.95)` scikit-learn chooses by itself the minimum `k` that reaches the threshold.
- **Elbow of the scree plot**: you look for the point after which every component adds little. Subjective, and on real data there is often no sharp elbow.
- **Cross-validation on the final task**: if PCA feeds a classifier, `k` is a hyperparameter like `C`. It is the criterion to prefer because it measures what you care about.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = {}
for k in [2, 5, 10, 20, 30, 40, 64]:
    model = make_pipeline(StandardScaler(), PCA(k), LogisticRegression(max_iter=3000))
    rows[k] = cross_val_score(model, X, y, cv=cv).mean()
print("cross-validated accuracy as a function of the components kept:")
print(pd.Series(rows).round(3).to_string())

With 20-30 components (a fraction of the 64 pixels) the accuracy is already almost that with all the components, and with 64 components (no reduction) it gains nothing: **the reduction removes noise and redundancy at no cost**.
The point where the curve flattens is the one you care about, and it does not necessarily coincide with 95% of the variance.

## How much variance is enough?

The explained variance measures the fidelity of the **reconstruction**, not the ability to discriminate the classes. Components with little variance can contain the information the target needs (C1). That is why thresholds on the variance are a starting point to
explore, not a selection criterion.

> **Common mistake.** Choosing `k` by looking at the explained variance on the whole dataset and then evaluating the model in cross-validation with that `k`: PCA was learned on the validation folds too. Put PCA in the `Pipeline` and search `k` with `GridSearchCV`.

## Check your understanding

<details><summary>What happens with `PCA(n_components=0.95)`?</summary>

With a number between 0 and 1, scikit-learn keeps the minimum number of components that explain at least that fraction of variance. It is convenient, but the number of components becomes dependent on the training data and changes from one fold to another: in cross-validation it is part of the model.

</details>

<details><summary>Do I have to keep the components with explained variance greater than 1 (Kaiser rule)?</summary>

The Kaiser rule makes sense only on standardized data (a component with variance below 1 explains less than a single original variable). It is a shortcut: on data with many redundant columns it tends to keep too many or too few components. Prefer cross-validation.

</details>